# UNH Aquafort Buoy Station - Initial Data Exploration

**Date:** October 31, 2025  
**Data Source:** UNH Aquafort Buoy Station (Campbell Scientific TOA5 Format)

## Overview

This notebook performs an initial exploratory data analysis (EDA) of two datasets from the UNH Aquafort buoy station:

1. **EXO2SumData** - Water quality monitoring data (EXO2 sonde)
2. **ZCelEngData** - Current profiler and engineering data (ZCell ADCP)

Both files are in Campbell Scientific TOA5 format, a specialized CSV format used by Campbell Scientific data loggers.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

# Import from our imta_analytics package
from imta_analytics.data import load_toa5_file
from imta_analytics.data.loaders import apply_marine_quality_filters

warnings.filterwarnings('ignore')

# Set style preferences
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✓ Libraries imported successfully")
print("✓ Using imta_analytics package v0.1.0")

## 2. Data Loading

### Understanding TOA5 Format

Campbell Scientific TOA5 files have a specific structure:
- **Line 1:** File format metadata (station name, logger model, etc.)
- **Line 2:** Column headers
- **Line 3:** Units for each column
- **Line 4:** Sampling information (e.g., "Smp", "Avg" - this confuses pandas type inference!)
- **Line 5+:** Actual data

The `load_toa5_file()` function from the `imta_analytics` package handles this format correctly by:
1. Skipping all 4 header lines to avoid type inference issues
2. Explicitly converting all columns (except TIMESTAMP) to numeric types
3. Parsing TIMESTAMP as datetime objects
4. Returning metadata and units separately for reference

In [ ]:
# Define file paths
DATA_DIR = Path('../data/aquafort-buoy-station')
EXO2_FILE = DATA_DIR / 'UNH-G2000B_EXO2SumData.dat'
ZCEL_FILE = DATA_DIR / 'UNH-G2000B_ZCelEngData.dat'

print(f"Data directory: {DATA_DIR.absolute()}")
print(f"EXO2 file exists: {EXO2_FILE.exists()}")
print(f"ZCel file exists: {ZCEL_FILE.exists()}")

### Load Data Using imta_analytics Package

We're now using the `load_toa5_file()` function from the `imta_analytics.data` module. This function:
- Returns a tuple of (DataFrame, metadata dict, units list)
- Handles the tricky TOA5 format with proper type inference
- Explicitly converts all columns to appropriate types (numeric or datetime)
- Returns metadata about the logger and station
- Returns units for each column

This makes our code cleaner and reusable across multiple notebooks and eventually in a web application.

### Load EXO2 Water Quality Data

In [ ]:
# Load EXO2 data
exo2_df, exo2_metadata, exo2_units = load_toa5_file(EXO2_FILE)

print("EXO2 Metadata:")
for key, value in exo2_metadata.items():
    print(f"  {key}: {value}")
print(f"\nDataset shape: {exo2_df.shape}")
print(f"Date range: {exo2_df['TIMESTAMP'].min()} to {exo2_df['TIMESTAMP'].max()}")

In [ ]:
# Display column information with units
print("\nEXO2 Columns and Units:")
print("=" * 60)
for i, (col, unit) in enumerate(zip(exo2_df.columns, exo2_units)):
    print(f"{i+1:2d}. {col:20s} [{unit}]")

In [ ]:
# Display first few rows
print("\nFirst 5 rows of EXO2 data:")
exo2_df.head().T

### Load ZCell Current Profiler Data

In [ ]:
# Load ZCel data
zcel_df, zcel_metadata, zcel_units = load_toa5_file(ZCEL_FILE)

print("ZCell Metadata:")
for key, value in zcel_metadata.items():
    print(f"  {key}: {value}")
print(f"\nDataset shape: {zcel_df.shape}")
print(f"Date range: {zcel_df['TIMESTAMP'].min()} to {zcel_df['TIMESTAMP'].max()}")

In [ ]:
# Display column information (first 30 columns)
print("\nZCell Columns and Units (first 30):")
print("=" * 60)
for i, (col, unit) in enumerate(zip(zcel_df.columns[:30], zcel_units[:30])):
    print(f"{i+1:2d}. {col:20s} [{unit}]")
print(f"\n... and {len(zcel_df.columns) - 30} more columns")

In [ ]:
# Display first few rows (limited columns for readability)
print("\nFirst 5 rows of ZCell data (selected columns):")
display_cols = ['TIMESTAMP', 'RECORD', 'batt_volt', 'PTemp', 'ZCel_Batt', 
                'ZCel_Hdng', 'ZCel_Pitch', 'ZCel_Roll', 'ZCel_Press', 'ZCel_WTmp']
#zcel_df[display_cols].head().T
zcel_df.head().T

In [ ]:
# Check if values within columns are actually strings or numbers
print("Checking actual value types in ZCell columns:")
print("=" * 60)

sample_cols = ['RECORD', 'batt_volt', 'PTemp', 'ZCel_WTmp', 'ZCel_Press']
for col in sample_cols:
    value_types = zcel_df[col].map(type).value_counts()
    print(f"\n{col}:")
    print(value_types)
    print(f"  Sample values: {zcel_df[col].head(3).tolist()}")

## 3. Data Quality Assessment

### 3.1 EXO2 Water Quality Data

In [ ]:
# Basic statistics
print("EXO2 Data Info:")
print("=" * 80)
exo2_df.info()

In [ ]:
# Check for missing values
missing_exo2 = exo2_df.isnull().sum()
missing_exo2_pct = (missing_exo2 / len(exo2_df)) * 100

print("\nMissing Values in EXO2 Data:")
print("=" * 60)
missing_summary = pd.DataFrame({
    'Missing Count': missing_exo2,
    'Percentage': missing_exo2_pct
}).sort_values('Missing Count', ascending=False)

missing_summary[missing_summary['Missing Count'] > 0]

In [ ]:
# Check for duplicate timestamps
duplicate_timestamps = exo2_df['TIMESTAMP'].duplicated().sum()
print(f"\nDuplicate timestamps in EXO2: {duplicate_timestamps}")

if duplicate_timestamps > 0:
    print("\nExample duplicate timestamps:")
    display(exo2_df[exo2_df['TIMESTAMP'].duplicated(keep=False)].sort_values('TIMESTAMP').head(10))

In [ ]:
# Descriptive statistics for key water quality parameters
key_params = ['EXO2pH', 'EXO2Temp', 'EXO2Salinity', 'EXO2DO', 'EXO2Chlor', 'EXO2Turb', 'EXO2Depth']
print("\nDescriptive Statistics - Key Water Quality Parameters:")
exo2_df[key_params].describe()

### 3.2 ZCell Current Profiler Data

In [ ]:
# Basic statistics
print("ZCell Data Info:")
print("=" * 80)
zcel_df.info()

In [ ]:
# Check for missing values in engineering parameters
eng_params = ['batt_volt', 'PTemp', 'ZCel_Batt', 'ZCel_Hdng', 'ZCel_Pitch', 
              'ZCel_Roll', 'ZCel_Press', 'ZCel_WTmp']

missing_zcel = zcel_df[eng_params].isnull().sum()
missing_zcel_pct = (missing_zcel / len(zcel_df)) * 100

print("\nMissing Values in ZCell Engineering Data:")
print("=" * 60)
pd.DataFrame({
    'Missing Count': missing_zcel,
    'Percentage': missing_zcel_pct
}).sort_values('Missing Count', ascending=False)

In [ ]:
# Descriptive statistics for engineering parameters
print("\nDescriptive Statistics - Engineering Parameters:")
zcel_df[eng_params].describe().T

In [ ]:
# Analyze current profile structure
# Find all depth and current speed columns
depth_cols = [col for col in zcel_df.columns if col.startswith('ZCelDpth')]
speed_cols = [col for col in zcel_df.columns if col.startswith('CurrSpd')]
dir_cols = [col for col in zcel_df.columns if col.startswith('CurrDir')]

print(f"\nCurrent Profile Structure:")
print(f"  Depth bins: {len(depth_cols)}")
print(f"  Speed measurements: {len(speed_cols)}")
print(f"  Direction measurements: {len(dir_cols)}")

if len(depth_cols) > 0:
    print(f"\nDepth range (first record):")
    first_depths = zcel_df[depth_cols].iloc[0]
    print(f"  Surface: {first_depths.iloc[0]} m")
    print(f"  Bottom: {first_depths.iloc[-1]} m")

## 4. Temporal Analysis

### 4.1 Time Series Coverage

In [ ]:
# Analyze sampling frequency
exo2_df['time_diff'] = exo2_df['TIMESTAMP'].diff()
zcel_df['time_diff'] = zcel_df['TIMESTAMP'].diff()

print("Sampling Frequency Analysis:")
print("=" * 60)
print(f"\nEXO2 Data:")
print(f"  Most common interval: {exo2_df['time_diff'].mode().iloc[0]}")
print(f"  Mean interval: {exo2_df['time_diff'].mean()}")
print(f"  Total duration: {exo2_df['TIMESTAMP'].max() - exo2_df['TIMESTAMP'].min()}")
print(f"  Number of records: {len(exo2_df)}")

print(f"\nZCell Data:")
print(f"  Most common interval: {zcel_df['time_diff'].mode().iloc[0]}")
print(f"  Mean interval: {zcel_df['time_diff'].mean()}")
print(f"  Total duration: {zcel_df['TIMESTAMP'].max() - zcel_df['TIMESTAMP'].min()}")
print(f"  Number of records: {len(zcel_df)}")

In [ ]:
# Visualize data availability
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('EXO2 Water Quality Data', 'ZCell Current Profiler Data'),
    vertical_spacing=0.15
)

# EXO2 availability
exo2_daily = exo2_df.set_index('TIMESTAMP').resample('H').count()['RECORD']
fig.add_trace(
    go.Scatter(x=exo2_daily.index, y=exo2_daily.values, 
               mode='lines', name='EXO2 Records/Hour',
               line=dict(color='blue')),
    row=1, col=1
)

# ZCell availability
zcel_daily = zcel_df.set_index('TIMESTAMP').resample('H').count()['RECORD']
fig.add_trace(
    go.Scatter(x=zcel_daily.index, y=zcel_daily.values,
               mode='lines', name='ZCell Records/Hour',
               line=dict(color='green')),
    row=2, col=1
)

fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Records/Hour", row=1, col=1)
fig.update_yaxes(title_text="Records/Hour", row=2, col=1)

fig.update_layout(height=600, title_text="Data Availability Timeline", showlegend=True)
fig.show()

### 4.2 Water Quality Time Series

In [ ]:
# Create interactive time series plots for key water quality parameters
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=('Temperature', 'Salinity', 'pH', 'Dissolved Oxygen (% Sat)',
                    'Chlorophyll', 'Turbidity', 'Depth', 'Specific Conductivity'),
    vertical_spacing=0.08,
    horizontal_spacing=0.1
)

params = [
    ('EXO2Temp', 1, 1, '°C'),
    ('EXO2Salinity', 1, 2, 'psu'),
    ('EXO2pH', 2, 1, ''),
    ('EXO2DO', 2, 2, '% Sat'),
    ('EXO2Chlor', 3, 1, 'RFU'),
    ('EXO2Turb', 3, 2, 'FNU'),
    ('EXO2Depth', 4, 1, 'm'),
    ('EXO2spCond', 4, 2, 'mS/cm')
]

for param, row, col, unit in params:
    fig.add_trace(
        go.Scatter(x=exo2_df['TIMESTAMP'], y=exo2_df[param],
                   mode='lines', name=param,
                   line=dict(width=1)),
        row=row, col=col
    )
    fig.update_yaxes(title_text=unit, row=row, col=col)

fig.update_layout(
    height=1200,
    title_text="Water Quality Parameters Over Time",
    showlegend=False
)
fig.show()

In [ ]:
# Let's check for outliers and anomalies in the water quality data
print("Checking for extreme values / outliers:")
print("=" * 80)

key_params = ['EXO2Temp', 'EXO2Salinity', 'EXO2pH', 'EXO2DO', 
              'EXO2Chlor', 'EXO2Turb', 'EXO2Depth', 'EXO2spCond']

for param in key_params:
    data = exo2_df[param].dropna()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 3 * iqr
    upper_bound = q3 + 3 * iqr
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    print(f"\n{param}:")
    print(f"  Range: {data.min():.3f} to {data.max():.3f}")
    print(f"  Median: {data.median():.3f}")
    print(f"  Q1/Q3: {q1:.3f} / {q3:.3f}")
    print(f"  Outliers (>3*IQR): {len(outliers)} values ({len(outliers)/len(data)*100:.2f}%)")
    if len(outliers) > 0:
        print(f"  Outlier range: {outliers.min():.3f} to {outliers.max():.3f}")
        print(f"  Example outlier values: {outliers.head(5).tolist()}")

**Issue Found:** Several parameters contain sensor error codes (143052, 193039, 91625, -86.48) that are breaking visualizations. These need to be filtered out as invalid readings.

In [ ]:
# Create a cleaned version of the data by filtering out sensor error codes
# Common Campbell Scientific error codes: 6999, 7999, NAN (which becomes a large number)
# We'll use reasonable physical bounds for each parameter

exo2_clean = exo2_df.copy()

# Define reasonable physical bounds for marine water quality parameters
bounds = {
    'EXO2Temp': (-2, 35),        # °C - freezing to tropical
    'EXO2Salinity': (0, 40),     # psu - fresh to hypersaline
    'EXO2pH': (6.5, 9.5),        # pH units
    'EXO2DO': (0, 200),          # % saturation
    'EXO2Chlor': (-1, 100),      # RFU - chlorophyll fluorescence
    'EXO2Turb': (0, 100),        # FNU - turbidity
    'EXO2Depth': (-1, 100),      # m - depth
    'EXO2spCond': (0, 70),       # mS/cm - specific conductivity
    'EXO2DomgpL': (0, 20)        # mg/L - dissolved oxygen
}

# Apply bounds and replace out-of-range values with NaN
for param, (lower, upper) in bounds.items():
    if param in exo2_clean.columns:
        mask = (exo2_clean[param] < lower) | (exo2_clean[param] > upper)
        invalid_count = mask.sum()
        if invalid_count > 0:
            print(f"{param}: Removed {invalid_count} invalid values ({invalid_count/len(exo2_clean)*100:.2f}%)")
            exo2_clean.loc[mask, param] = np.nan

print(f"\nCleaned dataset: {len(exo2_clean)} records")
print(f"Records with at least one valid measurement: {exo2_clean[list(bounds.keys())].notna().any(axis=1).sum()}")

In [ ]:
# Re-create time series plots with cleaned data
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=('Temperature', 'Salinity', 'pH', 'Dissolved Oxygen (% Sat)',
                    'Chlorophyll', 'Turbidity', 'Depth', 'Specific Conductivity'),
    vertical_spacing=0.08,
    horizontal_spacing=0.1
)

params = [
    ('EXO2Temp', 1, 1, '°C'),
    ('EXO2Salinity', 1, 2, 'psu'),
    ('EXO2pH', 2, 1, ''),
    ('EXO2DO', 2, 2, '% Sat'),
    ('EXO2Chlor', 3, 1, 'RFU'),
    ('EXO2Turb', 3, 2, 'FNU'),
    ('EXO2Depth', 4, 1, 'm'),
    ('EXO2spCond', 4, 2, 'mS/cm')
]

for param, row, col, unit in params:
    fig.add_trace(
        go.Scatter(x=exo2_clean['TIMESTAMP'], y=exo2_clean[param],
                   mode='lines', name=param,
                   line=dict(width=1)),
        row=row, col=col
    )
    fig.update_yaxes(title_text=unit, row=row, col=col)

fig.update_layout(
    height=1200,
    title_text="Water Quality Parameters Over Time (Cleaned Data)",
    showlegend=False
)
fig.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix for water quality parameters
wq_params = ['EXO2pH', 'EXO2Temp', 'EXO2spCond', 'EXO2Salinity', 
             'EXO2Turb', 'EXO2Chlor', 'EXO2DO', 'EXO2DomgpL', 'EXO2Depth']

correlation_matrix = exo2_df[wq_params].corr()

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.values,
    texttemplate='%{text:.2f}',
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Water Quality Parameters Correlation Matrix',
    width=800,
    height=700,
    xaxis={'side': 'bottom'}
)
fig.show()

In [ ]:
# Identify strongest correlations
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_pairs.append({
            'Param 1': correlation_matrix.columns[i],
            'Param 2': correlation_matrix.columns[j],
            'Correlation': correlation_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', key=abs, ascending=False)
print("\nStrongest Correlations (|r| > 0.7):")
print("=" * 60)
corr_df[abs(corr_df['Correlation']) > 0.7].head(10)

## 6. Distribution Analysis

In [ ]:
# Create distribution plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Distribution of Water Quality Parameters', fontsize=16, y=1.00)

plot_params = ['EXO2pH', 'EXO2Temp', 'EXO2Salinity', 'EXO2DO', 
               'EXO2Chlor', 'EXO2Turb', 'EXO2spCond', 'EXO2Depth', 'EXO2DomgpL']

for idx, param in enumerate(plot_params):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    # Histogram with KDE
    data = exo2_df[param].dropna()
    ax.hist(data, bins=50, alpha=0.7, edgecolor='black', density=True)
    
    # Add KDE line
    data.plot.kde(ax=ax, color='red', linewidth=2)
    
    ax.set_xlabel(param)
    ax.set_ylabel('Density')
    ax.set_title(f'{param}\n(μ={data.mean():.2f}, σ={data.std():.2f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Current Profile Analysis

In [ ]:
# Analyze a sample current profile
sample_idx = len(zcel_df) // 2  # Middle record
sample_record = zcel_df.iloc[sample_idx]

# Extract depth, speed, and direction for this record
depths = [sample_record[col] for col in depth_cols]
speeds = [sample_record[col] for col in speed_cols]
directions = [sample_record[col] for col in dir_cols]

print(f"Sample Current Profile (Record {sample_idx}):")
print(f"Timestamp: {sample_record['TIMESTAMP']}")
print(f"Water Temperature: {sample_record['ZCel_WTmp']:.2f}°C")
print(f"Pressure: {sample_record['ZCel_Press']:.3f} dBar")

# Create profile plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Current Speed Profile', 'Current Direction Profile')
)

fig.add_trace(
    go.Scatter(x=speeds, y=depths, mode='lines+markers',
               name='Speed', line=dict(color='blue', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=directions, y=depths, mode='lines+markers',
               name='Direction', line=dict(color='red', width=2)),
    row=1, col=2
)

fig.update_xaxes(title_text="Speed (m/s)", row=1, col=1)
fig.update_xaxes(title_text="Direction (°)", row=1, col=2)
fig.update_yaxes(title_text="Depth (m)", autorange='reversed', row=1, col=1)
fig.update_yaxes(title_text="Depth (m)", autorange='reversed', row=1, col=2)

fig.update_layout(
    height=500,
    title_text=f"Sample Current Profile - {sample_record['TIMESTAMP']}",
    showlegend=False
)
fig.show()

## 8. Summary Statistics and Data Quality Report

In [ ]:
print("=" * 80)
print("DATA QUALITY SUMMARY REPORT")
print("=" * 80)

print("\n1. EXO2 WATER QUALITY DATA")
print("-" * 80)
print(f"   Station: {exo2_metadata['station']}")
print(f"   Table: {exo2_metadata['table_name']}")
print(f"   Total Records: {len(exo2_df):,}")
print(f"   Date Range: {exo2_df['TIMESTAMP'].min()} to {exo2_df['TIMESTAMP'].max()}")
print(f"   Duration: {exo2_df['TIMESTAMP'].max() - exo2_df['TIMESTAMP'].min()}")
print(f"   Sampling Interval: {exo2_df['time_diff'].mode().iloc[0]} (most common)")
print(f"   Duplicate Timestamps: {duplicate_timestamps}")
print(f"   Columns: {len(exo2_df.columns)}")
print(f"   Missing Data: {exo2_df.isnull().sum().sum():,} values ({(exo2_df.isnull().sum().sum()/(len(exo2_df)*len(exo2_df.columns))*100):.2f}%)")

print("\n2. ZCEL CURRENT PROFILER DATA")
print("-" * 80)
print(f"   Station: {zcel_metadata['station']}")
print(f"   Table: {zcel_metadata['table_name']}")
print(f"   Total Records: {len(zcel_df):,}")
print(f"   Date Range: {zcel_df['TIMESTAMP'].min()} to {zcel_df['TIMESTAMP'].max()}")
print(f"   Duration: {zcel_df['TIMESTAMP'].max() - zcel_df['TIMESTAMP'].min()}")
print(f"   Sampling Interval: {zcel_df['time_diff'].mode().iloc[0]} (most common)")
print(f"   Columns: {len(zcel_df.columns)}")
print(f"   Depth Bins: {len(depth_cols)}")
print(f"   Missing Data: {zcel_df.isnull().sum().sum():,} values ({(zcel_df.isnull().sum().sum()/(len(zcel_df)*len(zcel_df.columns))*100):.2f}%)")

print("\n3. KEY FINDINGS")
print("-" * 80)
print(f"   • Both datasets are from the same station: {exo2_metadata['station']}")
print(f"   • Data format: Campbell Scientific TOA5")
print(f"   • Python modules suitable for loading: pandas (with custom TOA5 parser)")
print(f"   • Water quality measurements: pH, temperature, salinity, DO, chlorophyll, turbidity")
print(f"   • Current profiling: {len(depth_cols)} depth bins with speed and direction")
print(f"   • Time alignment: Both datasets cover similar time periods")
print("\n=" * 80)

## 9. Next Steps

Based on this initial exploration, recommended next steps include:

1. **Data Cleaning:**
   - Handle duplicate timestamps in EXO2 data
   - Investigate and address any anomalous values
   - Create flags for data quality issues

2. **Advanced Analysis:**
   - Diurnal/tidal patterns in water quality
   - Relationship between currents and water quality
   - Stratification analysis using temperature/salinity profiles
   - Seasonal trends (if longer time series available)

3. **Feature Engineering:**
   - Derived parameters (e.g., water density, Richardson number)
   - Tidal phase indicators
   - Current magnitude and vector components

4. **Modeling Opportunities:**
   - Predictive models for dissolved oxygen
   - Water quality forecasting
   - Current pattern classification
   - Anomaly detection for aquaculture monitoring

## 10. Export Processed Data (Optional)

In [ ]:
# Optionally export cleaned data to parquet for faster loading
# Uncomment to execute:

# output_dir = Path('../data/processed')
# output_dir.mkdir(exist_ok=True)

# exo2_df.to_parquet(output_dir / 'exo2_data.parquet', index=False)
# zcel_df.to_parquet(output_dir / 'zcel_data.parquet', index=False)

# print("✓ Data exported to parquet format")

print("Data export code available (commented out)")